# clustering tiendas

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = '/content/drive/MyDrive/data_dsmarket/'
except ImportError:
    DATA_PATH = 'data_dsmarket/'


df = pd.read_feather(DATA_PATH + 'df_preprocessed.feather')

Mounted at /content/drive


In [4]:
print(df.columns.tolist())
print(df.shape)

['id', 'item', 'category', 'department', 'store', 'store_code', 'region', 'd', 'sales', 'date', 'weekday', 'event', 'yearweek', 'sell_price', 'season', 'ingresos', 'is_holiday']
(58327370, 17)


## FEATURE ENGINEERING

In [5]:
df_tiendas = df.groupby('store_code').agg(
    ventas_totales   = ('sales', 'sum'),
    precio_medio     = ('sell_price', 'mean'),
    ingresos_totales = ('ingresos', 'sum')
).reset_index()

info = df[['store_code', 'store', 'region']].drop_duplicates()
df_tiendas = df_tiendas.merge(info, on='store_code', how='left')

print(df_tiendas.shape)
df_tiendas

/tmp/ipykernel_18195/1411655005.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_tiendas = df.groupby('store_code').agg(


(10, 6)


,store_code,ventas_totales,precio_medio,ingresos_totales,store,region
0,BOS_1,5595292,5.525140,19341454.0,South_End,Boston
1,BOS_2,7214384,5.515620,25264736.0,Roxbury,Boston
2,BOS_3,6089330,5.541304,21944200.0,Back_Bay,Boston
3,NYC_1,7698216,5.575351,27733958.0,Greenwich_Village,New York
4,NYC_2,5685475,5.579885,21507232.0,Harlem,New York
5,NYC_3,11188180,5.542587,39496760.0,Tribeca,New York
6,NYC_4,4103676,5.574479,15046231.0,Brooklyn,New York
7,PHI_1,5149062,5.588015,18233332.0,Midtown_Village,Philadelphia
8,PHI_2,6544012,5.605048,21656494.0,Yorktown,Philadelphia
9,PHI_3,6427782,5.579928,20751558.0,Queen_Village,Philadelphia


In [12]:
porcentajes_categorias = df.groupby(['store_code', 'category'])['sales'].sum().unstack(fill_value=0).reset_index()

/tmp/ipykernel_18195/2856437190.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  porcentajes_categorias = df.groupby(['store_code', 'category'])['sales'].sum().unstack(fill_value=0).reset_index()


In [14]:
# porcentajes_categorias = df.groupby(['store_code', 'category'])['sales'].sum().unstack(fill_value=0)
# porcentajes_categorias = porcentajes_categorias.div(porcentajes_categorias.sum(axis=1), axis=0)
# porcentajes_categorias

In [13]:
df_tiendas = df_tiendas.merge(porcentajes_categorias, on='store_code', how='left')

In [15]:
df_tiendas

,store_code,ventas_totales,precio_medio,ingresos_totales,store,region,ACCESORIES,HOME_&_GARDEN,SUPERMARKET
0,BOS_1,5595292,5.525140,19341454.0,South_End,Boston,429084,1387999,3778209
1,BOS_2,7214384,5.515620,25264736.0,Roxbury,Boston,635997,1563263,5015124
2,BOS_3,6089330,5.541304,21944200.0,Back_Bay,Boston,527258,1398984,4163088
3,NYC_1,7698216,5.575351,27733958.0,Greenwich_Village,New York,876678,1440710,5380828
4,NYC_2,5685475,5.579885,21507232.0,Harlem,New York,637803,1567413,3480259
5,NYC_3,11188180,5.542587,39496760.0,Tribeca,New York,960947,2711443,7515790
6,NYC_4,4103676,5.574479,15046231.0,Brooklyn,New York,564455,719796,2819425
7,PHI_1,5149062,5.588015,18233332.0,Midtown_Village,Philadelphia,655696,1055014,3438352
8,PHI_2,6544012,5.605048,21656494.0,Yorktown,Philadelphia,370214,1405614,4768184
9,PHI_3,6427782,5.579928,20751558.0,Queen_Village,Philadelphia,466668,1230434,4730680


In [16]:
transacciones = df[df['sales'] > 0].groupby('store_code')['ingresos'].sum()
ventas_con_compra = df[df['sales'] > 0].groupby('store_code')['sales'].sum()
df_tiendas['preciomedio'] = df_tiendas['store_code'].map(transacciones / ventas_con_compra)

/tmp/ipykernel_18195/20076724.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  transacciones = df[df['sales'] > 0].groupby('store_code')['ingresos'].sum()
/tmp/ipykernel_18195/20076724.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ventas_con_compra = df[df['sales'] > 0].groupby('store_code')['sales'].sum()


In [17]:
porcentajes_estaciones = df.groupby(['store_code', 'season'])['sales'].sum().unstack(fill_value=0)
porcentajes_estaciones = porcentajes_estaciones.div(porcentajes_estaciones.sum(axis=1), axis=0)
porcentajes_estaciones

/tmp/ipykernel_18195/270606927.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  porcentajes_estaciones = df.groupby(['store_code', 'season'])['sales'].sum().unstack(fill_value=0)


season,Invierno,Otoño,Primavera,Verano
store_code,,,,
BOS_1,0.244539,0.235342,0.269237,0.250883
BOS_2,0.245534,0.236748,0.265776,0.251942
BOS_3,0.244193,0.239163,0.265226,0.251418
NYC_1,0.243317,0.238041,0.266842,0.251801
NYC_2,0.247425,0.241674,0.261604,0.249298
NYC_3,0.241096,0.241038,0.263627,0.254240
NYC_4,0.245616,0.242869,0.267236,0.244279
PHI_1,0.265095,0.233043,0.265849,0.236013
PHI_2,0.261113,0.243208,0.255705,0.239974


In [18]:
df_tiendas = df_tiendas.merge(porcentajes_estaciones, on='store_code', how='left')

In [19]:
df_tiendas

,store_code,ventas_totales,precio_medio,ingresos_totales,store,region,ACCESORIES,HOME_&_GARDEN,SUPERMARKET,preciomedio,Invierno,Otoño,Primavera,Verano
0,BOS_1,5595292,5.525140,19341454.0,South_End,Boston,429084,1387999,3778209,3.456737,0.244539,0.235342,0.269237,0.250883
1,BOS_2,7214384,5.515620,25264736.0,Roxbury,Boston,635997,1563263,5015124,3.501995,0.245534,0.236748,0.265776,0.251942
2,BOS_3,6089330,5.541304,21944200.0,Back_Bay,Boston,527258,1398984,4163088,3.603713,0.244193,0.239163,0.265226,0.251418
3,NYC_1,7698216,5.575351,27733958.0,Greenwich_Village,New York,876678,1440710,5380828,3.602647,0.243317,0.238041,0.266842,0.251801
4,NYC_2,5685475,5.579885,21507232.0,Harlem,New York,637803,1567413,3480259,3.782838,0.247425,0.241674,0.261604,0.249298
5,NYC_3,11188180,5.542587,39496760.0,Tribeca,New York,960947,2711443,7515790,3.530222,0.241096,0.241038,0.263627,0.254240
6,NYC_4,4103676,5.574479,15046231.0,Brooklyn,New York,564455,719796,2819425,3.666525,0.245616,0.242869,0.267236,0.244279
7,PHI_1,5149062,5.588015,18233332.0,Midtown_Village,Philadelphia,655696,1055014,3438352,3.541098,0.265095,0.233043,0.265849,0.236013
8,PHI_2,6544012,5.605048,21656494.0,Yorktown,Philadelphia,370214,1405614,4768184,3.309360,0.261113,0.243208,0.255705,0.239974
9,PHI_3,6427782,5.579928,20751558.0,Queen_Village,Philadelphia,466668,1230434,4730680,3.228417,0.258498,0.237018,0.267199,0.237286


In [20]:
ventas_totales = df.groupby('store_code')['sales'].sum()
ventas_festivos = df[df['is_holiday'] == 1].groupby('store_code')['sales'].sum()
df_tiendas['pct_festivos'] = (ventas_festivos / ventas_totales).values

/tmp/ipykernel_18195/3624743555.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ventas_totales = df.groupby('store_code')['sales'].sum()
/tmp/ipykernel_18195/3624743555.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ventas_festivos = df[df['is_holiday'] == 1].groupby('store_code')['sales'].sum()


In [21]:
df['is_weekend'] = df['weekday'].isin(['Saturday', 'Sunday']).astype(int)

ventas_finde = df[df['is_weekend'] == 1].groupby('store_code')['sales'].sum()

df_tiendas['pct_finde'] = (ventas_finde / ventas_totales).values

/tmp/ipykernel_18195/723838815.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ventas_finde = df[df['is_weekend'] == 1].groupby('store_code')['sales'].sum()


In [22]:
dias_totales = df.groupby('store_code')['date'].nunique()
dias_sin_ventas = df[df['sales'] == 0].groupby('store_code')['date'].nunique()

df_tiendas['tasa_dias_sin_venta'] = (dias_sin_ventas / dias_totales).values

/tmp/ipykernel_18195/3496333291.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dias_totales = df.groupby('store_code')['date'].nunique()
/tmp/ipykernel_18195/3496333291.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dias_sin_ventas = df[df['sales'] == 0].groupby('store_code')['date'].nunique()


In [23]:
ventas_inicio = df[df['date'].dt.year.isin([2011, 2012])].groupby('store_code')['sales'].sum()
ventas_final = df[df['date'].dt.year.isin([2015, 2016])].groupby('store_code')['sales'].sum()

df_tiendas['tendencia'] = (ventas_final / ventas_inicio).values

/tmp/ipykernel_18195/1212787162.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ventas_inicio = df[df['date'].dt.year.isin([2011, 2012])].groupby('store_code')['sales'].sum()
/tmp/ipykernel_18195/1212787162.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ventas_final = df[df['date'].dt.year.isin([2015, 2016])].groupby('store_code')['sales'].sum()


In [24]:
media_festivos = df[df['is_holiday'] == 1].groupby('store_code')['sales'].mean()
media_no_festivos = df[df['is_holiday'] == 0].groupby('store_code')['sales'].mean()

df_tiendas['ratio_festivos'] = (media_festivos / media_no_festivos).values

/tmp/ipykernel_18195/972231654.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  media_festivos = df[df['is_holiday'] == 1].groupby('store_code')['sales'].mean()
/tmp/ipykernel_18195/972231654.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  media_no_festivos = df[df['is_holiday'] == 0].groupby('store_code')['sales'].mean()


In [25]:
df_dispersion = df.groupby('store_code').agg(
    std_sales  = ('sales', 'std'),
    max_sales  = ('sales', 'max'),
    min_sales  = ('sales', 'min'),
    mean_sales = ('sales', 'mean')
).reset_index()

df_tiendas = df_tiendas.merge(df_dispersion, on='store_code', how='left')

/tmp/ipykernel_18195/3073694380.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_dispersion = df.groupby('store_code').agg(


In [26]:
df_tiendas

,store_code,ventas_totales,precio_medio,ingresos_totales,store,region,ACCESORIES,HOME_&_GARDEN,SUPERMARKET,preciomedio,...,Verano,pct_festivos,pct_finde,tasa_dias_sin_venta,tendencia,ratio_festivos,std_sales,max_sales,min_sales,mean_sales
0,BOS_1,5595292,5.525140,19341454.0,South_End,Boston,429084,1387999,3778209,3.456737,...,0.250883,0.044695,0.348351,1.0,0.815860,0.958842,3.327232,634,0,0.959291
1,BOS_2,7214384,5.515620,25264736.0,Roxbury,Boston,635997,1563263,5015124,3.501995,...,0.251942,0.044093,0.346763,1.0,0.724142,0.945336,4.421267,626,0,1.236878
2,BOS_3,6089330,5.541304,21944200.0,Back_Bay,Boston,527258,1398984,4163088,3.603713,...,0.251418,0.043421,0.335612,1.0,0.917924,0.930270,3.796400,385,0,1.043992
3,NYC_1,7698216,5.575351,27733958.0,Greenwich_Village,New York,876678,1440710,5380828,3.602647,...,0.251801,0.042430,0.362848,1.0,0.857044,0.908113,4.058652,648,0,1.319829
4,NYC_2,5685475,5.579885,21507232.0,Harlem,New York,637803,1567413,3480259,3.782838,...,0.249298,0.043789,0.375393,1.0,0.912081,0.938517,2.759679,227,0,0.974753
5,NYC_3,11188180,5.542587,39496760.0,Tribeca,New York,960947,2711443,7515790,3.530222,...,0.254240,0.043302,0.337030,1.0,0.829562,0.927614,6.208486,763,0,1.918170
6,NYC_4,4103676,5.574479,15046231.0,Brooklyn,New York,564455,719796,2819425,3.666525,...,0.244279,0.042748,0.324281,1.0,0.922175,0.915218,2.004275,300,0,0.703559
7,PHI_1,5149062,5.588015,18233332.0,Midtown_Village,Philadelphia,655696,1055014,3438352,3.541098,...,0.236013,0.040891,0.360750,1.0,1.350346,0.873759,2.424396,224,0,0.882787
8,PHI_2,6544012,5.605048,21656494.0,Yorktown,Philadelphia,370214,1405614,4768184,3.309360,...,0.239974,0.040339,0.317730,1.0,1.323121,0.861478,3.866638,300,0,1.121945
9,PHI_3,6427782,5.579928,20751558.0,Queen_Village,Philadelphia,466668,1230434,4730680,3.228417,...,0.237286,0.041714,0.341205,1.0,0.647428,0.892115,4.065281,374,0,1.102018


In [27]:
print(df_tiendas.shape)
print(df_tiendas.columns.tolist())
print(df_tiendas.isnull().sum())

(10, 23)
['store_code', 'ventas_totales', 'precio_medio', 'ingresos_totales', 'store', 'region', 'ACCESORIES', 'HOME_&_GARDEN', 'SUPERMARKET', 'preciomedio', 'Invierno', 'Otoño', 'Primavera', 'Verano', 'pct_festivos', 'pct_finde', 'tasa_dias_sin_venta', 'tendencia', 'ratio_festivos', 'std_sales', 'max_sales', 'min_sales', 'mean_sales']
store_code             0
ventas_totales         0
precio_medio           0
ingresos_totales       0
store                  0
region                 0
ACCESORIES             0
HOME_&_GARDEN          0
SUPERMARKET            0
preciomedio            0
Invierno               0
Otoño                  0
Primavera              0
Verano                 0
pct_festivos           0
pct_finde              0
tasa_dias_sin_venta    0
tendencia              0
ratio_festivos         0
std_sales              0
max_sales              0
min_sales              0
mean_sales             0
dtype: int64


In [28]:
df_tiendas

,store_code,ventas_totales,precio_medio,ingresos_totales,store,region,ACCESORIES,HOME_&_GARDEN,SUPERMARKET,preciomedio,...,Verano,pct_festivos,pct_finde,tasa_dias_sin_venta,tendencia,ratio_festivos,std_sales,max_sales,min_sales,mean_sales
0,BOS_1,5595292,5.525140,19341454.0,South_End,Boston,429084,1387999,3778209,3.456737,...,0.250883,0.044695,0.348351,1.0,0.815860,0.958842,3.327232,634,0,0.959291
1,BOS_2,7214384,5.515620,25264736.0,Roxbury,Boston,635997,1563263,5015124,3.501995,...,0.251942,0.044093,0.346763,1.0,0.724142,0.945336,4.421267,626,0,1.236878
2,BOS_3,6089330,5.541304,21944200.0,Back_Bay,Boston,527258,1398984,4163088,3.603713,...,0.251418,0.043421,0.335612,1.0,0.917924,0.930270,3.796400,385,0,1.043992
3,NYC_1,7698216,5.575351,27733958.0,Greenwich_Village,New York,876678,1440710,5380828,3.602647,...,0.251801,0.042430,0.362848,1.0,0.857044,0.908113,4.058652,648,0,1.319829
4,NYC_2,5685475,5.579885,21507232.0,Harlem,New York,637803,1567413,3480259,3.782838,...,0.249298,0.043789,0.375393,1.0,0.912081,0.938517,2.759679,227,0,0.974753
5,NYC_3,11188180,5.542587,39496760.0,Tribeca,New York,960947,2711443,7515790,3.530222,...,0.254240,0.043302,0.337030,1.0,0.829562,0.927614,6.208486,763,0,1.918170
6,NYC_4,4103676,5.574479,15046231.0,Brooklyn,New York,564455,719796,2819425,3.666525,...,0.244279,0.042748,0.324281,1.0,0.922175,0.915218,2.004275,300,0,0.703559
7,PHI_1,5149062,5.588015,18233332.0,Midtown_Village,Philadelphia,655696,1055014,3438352,3.541098,...,0.236013,0.040891,0.360750,1.0,1.350346,0.873759,2.424396,224,0,0.882787
8,PHI_2,6544012,5.605048,21656494.0,Yorktown,Philadelphia,370214,1405614,4768184,3.309360,...,0.239974,0.040339,0.317730,1.0,1.323121,0.861478,3.866638,300,0,1.121945
9,PHI_3,6427782,5.579928,20751558.0,Queen_Village,Philadelphia,466668,1230434,4730680,3.228417,...,0.237286,0.041714,0.341205,1.0,0.647428,0.892115,4.065281,374,0,1.102018


In [35]:
# numericas = [col for col in df_tiendas.columns if col not in ['store_code', 'store', 'region']]
numericas = df_tiendas.select_dtypes(include='number').columns.tolist()

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import plotly.express as px

# Escalamos
X = df_tiendas[numericas].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Elbow
inertias = []
silhouettes = []
K = range(2, 8)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, kmeans.labels_))

# Plots
fig_elbow = px.line(x=list(K), y=inertias,
                    title='Elbow',
                    labels={'x': 'k', 'y': 'Inertia'})
fig_elbow.show()

fig_sil = px.line(x=list(K), y=silhouettes,
                  title='Silhouette',
                  labels={'x': 'k', 'y': 'Silhouette'})
fig_sil.show()

In [36]:
for k in [2, 3]:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    df_tiendas[f'cluster_k{k}'] = kmeans.fit_predict(X_scaled)
    print(f"\nk={k}:")
    print(df_tiendas[['store_code', 'store', 'region', f'cluster_k{k}']])


k=2:
  store_code              store        region  cluster_k2
0      BOS_1          South_End        Boston           0
1      BOS_2            Roxbury        Boston           0
2      BOS_3           Back_Bay        Boston           0
3      NYC_1  Greenwich_Village      New York           0
4      NYC_2             Harlem      New York           0
5      NYC_3            Tribeca      New York           0
6      NYC_4           Brooklyn      New York           1
7      PHI_1    Midtown_Village  Philadelphia           1
8      PHI_2           Yorktown  Philadelphia           1
9      PHI_3      Queen_Village  Philadelphia           1

k=3:
  store_code              store        region  cluster_k3
0      BOS_1          South_End        Boston           2
1      BOS_2            Roxbury        Boston           2
2      BOS_3           Back_Bay        Boston           2
3      NYC_1  Greenwich_Village      New York           0
4      NYC_2             Harlem      New York           2
5 

In [37]:
for k in [2, 3]:
    print(f"\n{'='*50}")
    print(f"k={k}")
    print(f"{'='*50}")

    # Tiendas por cluster
    display(df_tiendas.groupby(f'cluster_k{k}')[['store_code', 'store', 'region']].agg(list))

    # Media de features por cluster
    print("\nMedia de features por cluster:")
    display(df_tiendas.groupby(f'cluster_k{k}')[numericas].mean().T)


k=2


,store_code,store,region
cluster_k2,,,
0,"[BOS_1, BOS_2, BOS_3, NYC_1, NYC_2, NYC_3]","[South_End, Roxbury, Back_Bay, Greenwich_Villa...","[Boston, Boston, Boston, New York, New York, N..."
1,"[NYC_4, PHI_1, PHI_2, PHI_3]","[Brooklyn, Midtown_Village, Yorktown, Queen_Vi...","[New York, Philadelphia, Philadelphia, Philade..."



Media de features por cluster:


cluster_k2,0,1
ventas_totales,7.245146e+06,5.556133e+06
precio_medio,5.546648e+00,5.586867e+00
ingresos_totales,2.588139e+07,1.892190e+07
ACCESORIES,6.779612e+05,5.142582e+05
HOME_&_GARDEN,1.678302e+06,1.102714e+06
SUPERMARKET,4.888883e+06,3.939160e+06
Invierno,2.443504e-01,2.575803e-01
Otoño,2.386676e-01,2.390346e-01
Primavera,2.653851e-01,2.639971e-01
Verano,2.515970e-01,2.393879e-01



k=3


,store_code,store,region
cluster_k3,,,
0,"[NYC_1, NYC_3]","[Greenwich_Village, Tribeca]","[New York, New York]"
1,"[NYC_4, PHI_1, PHI_2, PHI_3]","[Brooklyn, Midtown_Village, Yorktown, Queen_Vi...","[New York, Philadelphia, Philadelphia, Philade..."
2,"[BOS_1, BOS_2, BOS_3, NYC_2]","[South_End, Roxbury, Back_Bay, Harlem]","[Boston, Boston, Boston, New York]"



Media de features por cluster:


cluster_k3,0,1,2
ventas_totales,9.443198e+06,5.556133e+06,6.146120e+06
precio_medio,5.558969e+00,5.586867e+00,5.540487e+00
ingresos_totales,3.361536e+07,1.892190e+07,2.201441e+07
ACCESORIES,9.188125e+05,5.142582e+05,5.575355e+05
HOME_&_GARDEN,2.076076e+06,1.102714e+06,1.479415e+06
SUPERMARKET,6.448309e+06,3.939160e+06,4.109170e+06
Invierno,2.422061e-01,2.575803e-01,2.454225e-01
Otoño,2.395392e-01,2.390346e-01,2.382318e-01
Primavera,2.652342e-01,2.639971e-01,2.654605e-01
Verano,2.530204e-01,2.393879e-01,2.508852e-01


In [38]:
df_tiendas.sort_values(by = 'SUPERMARKET', ascending = False)

,store_code,ventas_totales,precio_medio,ingresos_totales,store,region,ACCESORIES,HOME_&_GARDEN,SUPERMARKET,preciomedio,...,pct_finde,tasa_dias_sin_venta,tendencia,ratio_festivos,std_sales,max_sales,min_sales,mean_sales,cluster_k2,cluster_k3
5,NYC_3,11188180,5.542587,39496760.0,Tribeca,New York,960947,2711443,7515790,3.530222,...,0.337030,1.0,0.829562,0.927614,6.208486,763,0,1.918170,0,0
3,NYC_1,7698216,5.575351,27733958.0,Greenwich_Village,New York,876678,1440710,5380828,3.602647,...,0.362848,1.0,0.857044,0.908113,4.058652,648,0,1.319829,0,0
1,BOS_2,7214384,5.515620,25264736.0,Roxbury,Boston,635997,1563263,5015124,3.501995,...,0.346763,1.0,0.724142,0.945336,4.421267,626,0,1.236878,0,2
8,PHI_2,6544012,5.605048,21656494.0,Yorktown,Philadelphia,370214,1405614,4768184,3.309360,...,0.317730,1.0,1.323121,0.861478,3.866638,300,0,1.121945,1,1
9,PHI_3,6427782,5.579928,20751558.0,Queen_Village,Philadelphia,466668,1230434,4730680,3.228417,...,0.341205,1.0,0.647428,0.892115,4.065281,374,0,1.102018,1,1
2,BOS_3,6089330,5.541304,21944200.0,Back_Bay,Boston,527258,1398984,4163088,3.603713,...,0.335612,1.0,0.917924,0.930270,3.796400,385,0,1.043992,0,2
0,BOS_1,5595292,5.525140,19341454.0,South_End,Boston,429084,1387999,3778209,3.456737,...,0.348351,1.0,0.815860,0.958842,3.327232,634,0,0.959291,0,2
4,NYC_2,5685475,5.579885,21507232.0,Harlem,New York,637803,1567413,3480259,3.782838,...,0.375393,1.0,0.912081,0.938517,2.759679,227,0,0.974753,0,2
7,PHI_1,5149062,5.588015,18233332.0,Midtown_Village,Philadelphia,655696,1055014,3438352,3.541098,...,0.360750,1.0,1.350346,0.873759,2.424396,224,0,0.882787,1,1
6,NYC_4,4103676,5.574479,15046231.0,Brooklyn,New York,564455,719796,2819425,3.666525,...,0.324281,1.0,0.922175,0.915218,2.004275,300,0,0.703559,1,1


In [39]:
print(df_tiendas[numericas].std().sort_values(ascending=False))

ingresos_totales       6.742562e+06
ventas_totales         1.917808e+06
SUPERMARKET            1.328588e+06
HOME_&_GARDEN          5.124819e+05
ACCESORIES             1.878087e+05
max_sales              1.993810e+02
std_sales              1.184961e+00
cluster_k3             7.888106e-01
cluster_k2             5.163978e-01
mean_sales             3.288007e-01
tendencia              2.315228e-01
ratio_festivos         3.148937e-02
precio_medio           2.946683e-02
pct_finde              1.777553e-02
Invierno               8.534834e-03
Verano                 6.756005e-03
Primavera              3.830723e-03
Otoño                  3.373834e-03
pct_festivos           1.408750e-03
tasa_dias_sin_venta    0.000000e+00
min_sales              0.000000e+00
dtype: float64


In [40]:
numericas_revisadas = [
    'mean_sales',
    'std_sales',
    'tendencia',
    'ratio_festivos',
    'SUPERMARKET',
    'HOME_&_GARDEN',
    'ACCESORIES',
    'precio_medio',
    'pct_finde',
]

In [41]:
X_clean = df_tiendas[numericas_revisadas].copy()
scaler = StandardScaler()
X_clean_scaled = scaler.fit_transform(X_clean)

# Elbow + Silhouette
inertias = []
silhouettes = []
K = range(2, 8)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans.fit(X_clean_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_clean_scaled, kmeans.labels_))

fig_elbow = px.line(x=list(K), y=inertias,
                    title='Elbow Curve',
                    labels={'x': 'k', 'y': 'Inertia'})
fig_elbow.show()

fig_sil = px.line(x=list(K), y=silhouettes,
                  title='Silhouette Score',
                  labels={'x': 'k', 'y': 'Silhouette'})
fig_sil.show()

In [42]:
for k in [2,3, 4]:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    df_tiendas[f'cluster_k{k}'] = kmeans.fit_predict(X_clean_scaled)
    print(f"\nk={k}:")
    display(df_tiendas.groupby(f'cluster_k{k}')[['store_code', 'store', 'region']].agg(list))
    print("\nMedia de features por cluster:")
    display(df_tiendas.groupby(f'cluster_k{k}')[numericas_revisadas].mean().round(3).T)


k=2:


,store_code,store,region
cluster_k2,,,
0,"[BOS_2, NYC_1, NYC_3]","[Roxbury, Greenwich_Village, Tribeca]","[Boston, New York, New York]"
1,"[BOS_1, BOS_3, NYC_2, NYC_4, PHI_1, PHI_2, PHI_3]","[South_End, Back_Bay, Harlem, Brooklyn, Midtow...","[Boston, Boston, New York, New York, Philadelp..."



Media de features por cluster:


cluster_k2,0,1
mean_sales,1.492,0.970
std_sales,4.896,3.178
tendencia,0.804,0.984
ratio_festivos,0.927,0.910
SUPERMARKET,5970580.667,3882599.571
HOME_&_GARDEN,1905138.667,1252179.143
ACCESORIES,824540.667,521596.857
precio_medio,5.545,5.571
pct_finde,0.349,0.343



k=3:


...

cluster_k3,0,1,2
mean_sales,1.619,0.953,1.054
std_sales,5.134,3.090,3.576
tendencia,0.843,1.061,0.843
ratio_festivos,0.918,0.886,0.943
SUPERMARKET,6448309.000,3939160.250,4109170.000
HOME_&_GARDEN,2076076.500,1102714.500,1479414.750
ACCESORIES,918812.500,514258.250,557535.500
precio_medio,5.559,5.587,5.540
pct_finde,0.350,0.336,0.352



k=4:


,store_code,store,region
cluster_k4,,,
0,"[NYC_1, NYC_3]","[Greenwich_Village, Tribeca]","[New York, New York]"
1,"[NYC_4, PHI_3]","[Brooklyn, Queen_Village]","[New York, Philadelphia]"
2,"[BOS_1, BOS_2, BOS_3, NYC_2]","[South_End, Roxbury, Back_Bay, Harlem]","[Boston, Boston, Boston, New York]"
3,"[PHI_1, PHI_2]","[Midtown_Village, Yorktown]","[Philadelphia, Philadelphia]"



Media de features por cluster:


cluster_k4,0,1,2,3
mean_sales,1.619,0.903,1.054,1.002
std_sales,5.134,3.035,3.576,3.146
tendencia,0.843,0.785,0.843,1.337
ratio_festivos,0.918,0.904,0.943,0.868
SUPERMARKET,6448309.000,3775052.500,4109170.000,4103268.000
HOME_&_GARDEN,2076076.500,975115.000,1479414.750,1230314.000
ACCESORIES,918812.500,515561.500,557535.500,512955.000
precio_medio,5.559,5.577,5.540,5.597
pct_finde,0.350,0.333,0.352,0.339


In [43]:
df_tiendas[df_tiendas['cluster_k2'] == 0]

,store_code,ventas_totales,precio_medio,ingresos_totales,store,region,ACCESORIES,HOME_&_GARDEN,SUPERMARKET,preciomedio,...,tasa_dias_sin_venta,tendencia,ratio_festivos,std_sales,max_sales,min_sales,mean_sales,cluster_k2,cluster_k3,cluster_k4
1,BOS_2,7214384,5.515620,25264736.0,Roxbury,Boston,635997,1563263,5015124,3.501995,...,1.0,0.724142,0.945336,4.421267,626,0,1.236878,0,2,2
3,NYC_1,7698216,5.575351,27733958.0,Greenwich_Village,New York,876678,1440710,5380828,3.602647,...,1.0,0.857044,0.908113,4.058652,648,0,1.319829,0,0,0
5,NYC_3,11188180,5.542587,39496760.0,Tribeca,New York,960947,2711443,7515790,3.530222,...,1.0,0.829562,0.927614,6.208486,763,0,1.918170,0,0,0


In [44]:
df_tiendas[df_tiendas['cluster_k2'] == 1]

,store_code,ventas_totales,precio_medio,ingresos_totales,store,region,ACCESORIES,HOME_&_GARDEN,SUPERMARKET,preciomedio,...,tasa_dias_sin_venta,tendencia,ratio_festivos,std_sales,max_sales,min_sales,mean_sales,cluster_k2,cluster_k3,cluster_k4
0,BOS_1,5595292,5.525140,19341454.0,South_End,Boston,429084,1387999,3778209,3.456737,...,1.0,0.815860,0.958842,3.327232,634,0,0.959291,1,2,2
2,BOS_3,6089330,5.541304,21944200.0,Back_Bay,Boston,527258,1398984,4163088,3.603713,...,1.0,0.917924,0.930270,3.796400,385,0,1.043992,1,2,2
4,NYC_2,5685475,5.579885,21507232.0,Harlem,New York,637803,1567413,3480259,3.782838,...,1.0,0.912081,0.938517,2.759679,227,0,0.974753,1,2,2
6,NYC_4,4103676,5.574479,15046231.0,Brooklyn,New York,564455,719796,2819425,3.666525,...,1.0,0.922175,0.915218,2.004275,300,0,0.703559,1,1,1
7,PHI_1,5149062,5.588015,18233332.0,Midtown_Village,Philadelphia,655696,1055014,3438352,3.541098,...,1.0,1.350346,0.873759,2.424396,224,0,0.882787,1,1,3
8,PHI_2,6544012,5.605048,21656494.0,Yorktown,Philadelphia,370214,1405614,4768184,3.309360,...,1.0,1.323121,0.861478,3.866638,300,0,1.121945,1,1,3
9,PHI_3,6427782,5.579928,20751558.0,Queen_Village,Philadelphia,466668,1230434,4730680,3.228417,...,1.0,0.647428,0.892115,4.065281,374,0,1.102018,1,1,1


In [45]:
df_tiendas[df_tiendas['cluster_k3'] == 2]

,store_code,ventas_totales,precio_medio,ingresos_totales,store,region,ACCESORIES,HOME_&_GARDEN,SUPERMARKET,preciomedio,...,tasa_dias_sin_venta,tendencia,ratio_festivos,std_sales,max_sales,min_sales,mean_sales,cluster_k2,cluster_k3,cluster_k4
0,BOS_1,5595292,5.525140,19341454.0,South_End,Boston,429084,1387999,3778209,3.456737,...,1.0,0.815860,0.958842,3.327232,634,0,0.959291,1,2,2
1,BOS_2,7214384,5.515620,25264736.0,Roxbury,Boston,635997,1563263,5015124,3.501995,...,1.0,0.724142,0.945336,4.421267,626,0,1.236878,0,2,2
2,BOS_3,6089330,5.541304,21944200.0,Back_Bay,Boston,527258,1398984,4163088,3.603713,...,1.0,0.917924,0.930270,3.796400,385,0,1.043992,1,2,2
4,NYC_2,5685475,5.579885,21507232.0,Harlem,New York,637803,1567413,3480259,3.782838,...,1.0,0.912081,0.938517,2.759679,227,0,0.974753,1,2,2
